[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/fast_track/12_tools_and_agents.ipynb)

# 📓 Notebook 12 (fast track) — Tool Calling and Small Agents

> **Module:** AI Engineering · **Estimated time:** 60–80 min · **Difficulty:** Intermediate

So far, an LLM has been a function that takes text and returns text — a brilliant but forgetful intern who could only *talk*. This notebook hands that intern a phone and a calculator. We give the model **tools** — functions it can decide to call — and watch it do multi-step work: look up data, do arithmetic, query a DataFrame, then summarise the result.

This single pattern is the backbone of **every modern AI product**: ChatGPT plugins, Cursor's code edits, Slack's AI summaries, every internal copilot. Once you understand the call → execute → return loop, you can build all of them.

By the end you will have built a small **data assistant** — a 100-line program that answers business questions about our support-ops dataset by running pandas queries on its own.

> 🛟 **Our running example: the support-ops data assistant.** Picture the analyst on your support team who fields questions all day — *"How many tickets did we get?"*, *"What's CSAT for Chat?"*, *"Summarise the Email channel."* Each answer means running a little pandas query. We'll build an assistant that takes the plain-English question, **decides which query to run**, runs it, and reports back. One dataset, a handful of tools, one question after another — that's our thread for the whole notebook.

> 🧭 **Mental model — the planner and the hands.** Keep two characters separate in your head. The **LLM is the planner**: it reads the question and *says* "I'd like to call `mean_satisfaction(channel='Chat')`" — but it can only ever produce *text*. **Your code is the hands**: it looks that tool name up in a registry and *actually runs the function*. An agent is just these two passing notes back and forth — planner asks, hands act, result goes back, repeat — until the planner says "I'm done." Every confusing thing about agents gets clearer when you ask: *was that the planner talking, or the hands acting?*

## 🎯 Learning objectives

By the end of this notebook you can:

1. Define a **tool** with a JSON schema the model can read.
2. Run the **call → execute → return** loop manually, step by step.
3. Build a multi-tool agent that picks the right tool for each question.
4. Add a **safety budget** (max steps, max tool calls) to prevent runaway loops.
5. Log every tool call so you can debug what the agent did.
6. Recognise when an agent is overkill — and when a single LLM call is enough.

## ✅ Prerequisites

Notebooks 10 (LLM workflows) and 11 (embeddings & retrieval). Familiarity with JSON (NB 9) helps.

> 🏎️ **You're on the fast track.** This is a trimmed version of the canonical [`06_ai_engineering/24_tools_and_agents.ipynb`](../06_ai_engineering/24_tools_and_agents.ipynb) — tool calling and small agents. The deepest Stretch exercises (A and B) and the 🎁 Bonus mini-project have been removed to keep the notebook lean; the harder Stretch C and D are kept. Open the canonical version once you want the deeper material.

---


## 1. The tool-calling loop in one picture

Before we wire up the data assistant, let's see the whole machine in one diagram — the planner and the hands passing notes back and forth.

```
   ┌───────────────────────────────────────────────────────────────┐
   │                                                                │
   │   user message                                                 │
   │       │                                                        │
   │       ▼                                                        │
   │   ┌───────┐                                                    │
   │   │  LLM  │── responds with: "call tool get_total_tickets()"   │
   │   └───┬───┘                                                    │
   │       │                                                        │
   │       ▼                                                        │
   │   your code looks up that tool name, runs it, captures result  │
   │       │                                                        │
   │       ▼                                                        │
   │   ┌───────┐                                                    │
   │   │  LLM  │── sees the result and: "now I can answer"          │
   │   └───┬───┘                                                    │
   │       │                                                        │
   │       ▼                                                        │
   │   final answer to user                                         │
   │                                                                │
   └────────────────────────────────────────────────────────────────┘
```

The "magic" is that the **planner** gets to **decide** which tool to call and with what arguments; the **hands** (your code) carry it out. Your job is to:

1. Describe each tool to the model (name + JSON-schema arguments).
2. Listen for tool-call responses.
3. Run the matching Python function.
4. Send the result back to the model.
5. Loop until the model produces a final answer.

### 🔬 What actually happens in the loop — the model never runs the tool

The picture above hides the single most-misunderstood fact about agents: **the LLM cannot run code.** It only emits *text*. "Tool use" is a convention layered on top of that text:

```
   YOU (planner side)                         YOUR CODE (hands side)
   ──────────────────                         ──────────────────────
   1. describe tools  ───────────────────────▶  messages = [...tools...]
                                                        │
   2. model emits a STRING that we parse  ◀────  fake_llm(messages)
      {"tool": "calc", "args": {...}}                   │
                                                        ▼
   3. YOUR python looks up "calc" in a dict  ──▶  registry["calc"](**args)
      and ACTUALLY runs the function                    │  → real result
                                                        ▼
   4. feed the result back as a new message  ──▶  messages.append(result)
                                                        │
   5. loop: model now has the answer  ◀──────── fake_llm(messages)
      {"final": "..."}  → DONE
```

> 🎯 **The split that matters.** The model is the **planner that asks**; your code is the **hands that act**. The model says *"I'd like to call `calc('21*2')`"* — it's your `registry["calc"]` line that does the multiplication. Nothing runs unless your loop chooses to run it.


Let's prove it with a fully offline mock — no API, stdlib only. Three pieces:

1. a **tool registry**: a plain `dict` mapping a tool *name* to a real Python function;
2. a **`fake_llm(messages)`**: a deterministic stand-in that reads the conversation and returns *either* a tool-call dict *or* a final answer (this is the part a real model replaces);
3. the **loop** that wires them together and prints every step.


In [ ]:
# ── 1. TOOL REGISTRY: name -> real Python function (YOUR code, the "hands") ──
def calc(expression: str) -> float:
    # A real function. The model can NEVER run this itself.
    return eval(expression, {"__builtins__": {}}, {})   # toy demo only

REGISTRY = {"calc": calc}

# ── 2. fake_llm: deterministic stand-in for a real model (the "planner") ──
# It returns TEXT we parse into one of two shapes:
#   {"tool": name, "args": {...}}   -> "please run this tool for me"
#   {"final": "..."}                -> "here is my answer, we're done"
def fake_llm(messages: list[dict]) -> dict:
    last = messages[-1]
    if last["role"] == "user":
        # Hasn't seen a tool result yet -> ask to use the calculator.
        return {"tool": "calc", "args": {"expression": "21 * 2"}}
    # Otherwise the last message is a tool result -> wrap it as the answer.
    return {"final": f"The answer is {last['content']}."}

print("registry tools:", list(REGISTRY))
print("calc IS a real fn:", calc("21 * 2"))   # 42 — runs because WE called it


In [ ]:
# ── 3. THE LOOP: ask -> run tool -> return result -> repeat ──
def agent(user_msg: str, max_steps: int = 5) -> str:
    messages = [{"role": "user", "content": user_msg}]
    for step in range(1, max_steps + 1):
        reply = fake_llm(messages)                      # PLANNER asks
        if "final" in reply:                            # model is done
            print(f"step {step}: 🏁 final  -> {reply['final']}")
            return reply["final"]
        # It's a tool call. YOUR code looks up the name and ACTUALLY runs it.
        name, args = reply["tool"], reply["args"]
        print(f"step {step}: 🤖 model asks -> {name}({args})")
        result = REGISTRY[name](**args)                 # HANDS act
        print(f"step {step}: 🛠️  your code ran it -> {result}")
        # Feed the result back so the model can use it next turn.
        messages.append({"role": "tool", "content": str(result)})
    return "⚠️ hit max_steps without a final answer"

agent("What is 21 times 2?")


## 2. Setup — a richer MockLLM that understands tools

Real APIs (OpenAI, Anthropic, etc.) have built-in tool-calling support. To stay offline and deterministic, we extend the `MockLLM` from NB 10 with a small "decision rule" that looks at the user message and decides whether to call a tool.

In production this would be replaced by **one line**: the real model just *does* this for you.

In [ ]:
import json
import re
import pandas as pd
import numpy as np
from typing import Any, Callable

# ---------------------------------------------------------------------
# A tiny offline LLM-with-tools — the same MockLLM idea as in llm_providers.py
# (introduced in NB 10), extended here with tool-calling and vendored inline so
# the notebook stays self-contained.
#
# Real models (OpenAI, Anthropic, Gemini) interpret tool schemas natively.
# Here we simulate the decision with simple keyword rules so the notebook
# runs without internet or an API key. The CONTRACT is identical:
#   - chat() returns either {"text": str} OR
#                          {"tool_call": {"name": str, "arguments": dict}}.
# Plug in a real provider and only the inside of chat() changes.
# ---------------------------------------------------------------------

class MockLLM:
    def __init__(self):
        self.calls = 0   # for cost-accounting demos

    def chat(self, messages: list[dict],
             tools: list[dict] | None = None,
             **_) -> dict:
        self.calls += 1
        user_msg = next((m["content"] for m in reversed(messages)
                          if m["role"] == "user"), "").lower()
        # If we already saw a tool result, produce the final answer.
        had_tool_result = any(m["role"] == "tool" for m in messages)

        if not tools or had_tool_result:
            return {"text": self._compose_final(messages)}

        # Heuristic tool routing — easy and replaceable
        for tool in tools:
            name = tool["name"]
            if self._should_call(name, user_msg):
                args = self._infer_args(tool, user_msg)
                return {"tool_call": {"name": name, "arguments": args}}

        # No tool seemed to match → just answer in text.
        return {"text": "I'm not sure how to answer that with the tools I have."}

    # ----- internal helpers --------------------------------------------------
    @staticmethod
    def _should_call(tool_name: str, user_msg: str) -> bool:
        rules = {
            "total_tickets":       ["how many", "total", "tickets"],
            "mean_satisfaction":   ["satisfaction", "csat", "happy"],
            "channel_summary":     ["summary", "summarise", "summarize", "overview", "report"],
            "filter_channel":      ["chat", "email", "phone", "web form", "social"],
            "calculator":          ["+", "-", "*", "/", "calculate", "compute"],
        }
        keywords = rules.get(tool_name, [])
        return any(kw in user_msg for kw in keywords)

    @staticmethod
    def _infer_args(tool: dict, user_msg: str) -> dict:
        # Fish out a channel name if one of the recognised words appears
        for ch in ("Chat", "Email", "Phone", "Web Form", "Social"):
            if ch.lower() in user_msg:
                return {"channel": ch} if "channel" in tool["parameters"]["properties"] else {}
        # Fish out a math expression for the calculator
        if tool["name"] == "calculator":
            m = re.search(r"([\-\d\.\+\*\/\(\)\s]+)", user_msg)
            return {"expression": (m.group(0).strip() if m else "0")}
        return {}

    @staticmethod
    def _compose_final(messages: list[dict]) -> str:
        # Find the last tool result in the conversation and verbalise it
        tool_outputs = [m for m in messages if m["role"] == "tool"]
        if tool_outputs:
            last = tool_outputs[-1]
            return f"Based on the tool '{last['name']}', the answer is: {last['content']}."
        return "(no tools were used; I have no answer.)"


llm = MockLLM()
print("MockLLM (tool-aware) ready ✅")


> 🔌 **Using a real provider (OpenAI / Anthropic / Gemini / Ollama).**
> Every notebook in this module uses an offline `MockLLM` by default so you can run them without internet or API keys. For the text-generation notebooks (NB 10–11) the swap is one line — the unified interface in [`llm_providers.py`](../llm_providers.py) lets you use **OpenAI**, **Anthropic** (Claude), **Google** (Gemini), or a **local model via Ollama** without changing anything else. This notebook is the exception: it vendors its own **tool-aware** MockLLM inline, so tool calling with a real provider goes through the provider SDK's native tool API (e.g. the `tools=` parameter in the OpenAI / Anthropic clients) rather than a one-line swap.
> See the [LLM Providers Guide](../06_ai_engineering/A1_llm_providers_guide.ipynb) for the swap-in instructions, model recommendations, and cost estimates.


> 🎯 **The contract.** Notice that `chat()` returns one of two shapes — either `{"text": ...}` (a final answer) or `{"tool_call": ...}` (a request for your code to run something). Every real tool-calling API in the wild uses this exact contract.

## 3. Tool schemas — what the model needs to know

Now we build the *actual* tools our support-ops assistant will use — `total_tickets`, `mean_satisfaction`, `channel_summary`, `calculator` — and tell the planner about them. The planner can only call a tool it knows exists, so the schema is how we hand it a menu.

A tool is described to the model with a tiny JSON schema. The schema serves three purposes:

1. **Tells the model the tool exists** (so it can choose to call it).
2. **Describes what the tool does** (the `description` field is what the model reads).
3. **Specifies the arguments** (so the model produces valid JSON).

In [ ]:
# Our toy support-ops dataset (same idea as NB 9)
np.random.seed(42)
channels = ["Email", "Chat", "Phone", "Web Form", "Social"]
support_ops = pd.DataFrame({
    "channel":          np.repeat(channels, 12),
    "month":            list(range(1, 13)) * 5,
    "tickets_total":    np.random.randint(1000, 8000, size=60),
    "automation_rate":  np.clip(np.random.normal(0.6, 0.15, 60), 0.1, 0.95),
    "satisfaction":     np.clip(np.random.normal(4.0, 0.2, 60), 1, 5),
    "cost_per_ticket":  np.random.uniform(0.3, 5.5, 60),
})

# A tiny "tool registry"
def total_tickets() -> int:
    """Sum the tickets_total column across all channels and months."""
    return int(support_ops["tickets_total"].sum())

def mean_satisfaction(channel: str = None) -> float:
    """Average satisfaction overall, or for a specific channel."""
    if channel:
        sub = support_ops[support_ops["channel"] == channel]
        if sub.empty:
            return float("nan")
        return float(sub["satisfaction"].mean())
    return float(support_ops["satisfaction"].mean())

def channel_summary(channel: str) -> dict:
    """Return key metrics for a single channel."""
    sub = support_ops[support_ops["channel"] == channel]
    return {
        "channel":         channel,
        "n_months":        len(sub),
        "total_tickets":   int(sub["tickets_total"].sum()),
        "mean_auto":       round(float(sub["automation_rate"].mean()), 3),
        "mean_csat":       round(float(sub["satisfaction"].mean()), 2),
        "mean_cost":       round(float(sub["cost_per_ticket"].mean()), 2),
    }

def calculator(expression: str) -> float:
    """Evaluate a basic arithmetic expression. Numeric / operators only."""
    if not re.fullmatch(r"[\d\.\+\-\*\/\(\)\s]+", expression):
        raise ValueError(f"Unsafe expression: {expression!r}")
    return eval(expression)   # safe: we already validated the chars


# The schema each tool advertises to the LLM
TOOLS = [
    {
        "name": "total_tickets",
        "description": "Return the total number of tickets across all channels and months.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "mean_satisfaction",
        "description": "Return the mean customer-satisfaction (CSAT) score, optionally for one channel.",
        "parameters": {
            "type": "object",
            "properties": {"channel": {"type": "string",
                                        "description": "Channel name, e.g. 'Chat'."}},
            "required": [],
        },
    },
    {
        "name": "channel_summary",
        "description": "Return a summary of key metrics for one channel.",
        "parameters": {
            "type": "object",
            "properties": {"channel": {"type": "string"}},
            "required": ["channel"],
        },
    },
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression (only digits and + - * / parentheses).",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
        },
    },
]
# Index from name → callable (your code's "router")
TOOL_FNS = {
    "total_tickets":     total_tickets,
    "mean_satisfaction": mean_satisfaction,
    "channel_summary":   channel_summary,
    "calculator":        calculator,
}
print(f"Registered {len(TOOL_FNS)} tools: {list(TOOL_FNS.keys())}")


> 💡 **Why JSON schemas?** Because the LLM needs to know what arguments to produce — and you need a way to validate the output. JSON Schema is the industry-standard format every major LLM API supports.

---

### ✋ Quick exercise (~2 min) — Add a tool schema

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The team wants the assistant to report automation rate per channel. Write the JSON schema dict for a tool `automation_summary` that takes one **required** string argument `channel`. Just the schema — no need to register or route it yet.

In [ ]:
# ✍️ Your turn 👇
automation_schema = {
    "name": "automation_summary",
    # add: description, parameters (a required string "channel")
}


<details>
<summary>✅ <b>Solution</b></summary>

```python
automation_schema = {
    "name": "automation_summary",
    "description": "Return the mean automation rate for one channel.",
    "parameters": {
        "type": "object",
        "properties": {"channel": {"type": "string",
                                    "description": "Channel name, e.g. 'Chat'."}},
        "required": ["channel"],
    },
}
```

A tool schema is just a `name`, a `description` the planner reads, and a JSON-Schema `parameters` block. Listing `channel` under `required` tells the model it must supply that argument.
</details>

## 4. One round of tool calling — manually

Let's walk through the loop step by step.

In [ ]:
# Step 1: user asks a question
user_question = "What's the mean satisfaction for Chat?"

# Step 2: send messages + tools to the LLM
messages = [
    {"role": "system",
     "content": "You are a data assistant. Use the available tools to answer questions about support_ops."},
    {"role": "user", "content": user_question},
]
response = llm.chat(messages=messages, tools=TOOLS)
print("LLM response:", response)


In [ ]:
# Step 3: if the model wants a tool, run it
if "tool_call" in response:
    call = response["tool_call"]
    print(f"\nLLM requested: {call['name']}({call['arguments']})")
    fn = TOOL_FNS[call["name"]]
    result = fn(**call["arguments"])
    print(f"Tool returned: {result}")

    # Step 4: feed the result back to the LLM as a 'tool' message
    messages.append({"role": "assistant", "content": "",
                     "tool_call": call})
    messages.append({"role": "tool", "name": call["name"],
                     "content": json.dumps(result)})

    # Step 5: ask the LLM again — now it can produce a final answer
    final = llm.chat(messages=messages, tools=TOOLS)
    print(f"\nFinal answer: {final.get('text')}")


> 🎯 **Three actors in the loop:** the user (whose question kicks things off), the LLM (which decides *what* to do), and your code (which actually *does* the thing). Tool calling is the contract between the second and third.

## 5. Wrapping the loop in a function

We've run one round by hand. But our analyst gets a *new* question every few minutes — so let's package the planner-and-hands dance into a reusable `run_agent` that handles any support-ops question. The five lines we just wrote will be the body of every agent we build. Let's put them in a function with a **safety budget**.

In [ ]:
def run_agent(user_question: str,
              tools_schema: list[dict],
              tool_registry: dict,
              max_steps: int = 5,
              verbose: bool = True) -> dict:
    """Run the call → execute → return loop until the LLM stops asking for tools."""
    messages = [
        {"role": "system",
         "content": "You are a data assistant. Use the available tools to answer questions."},
        {"role": "user", "content": user_question},
    ]
    log = []

    for step in range(1, max_steps + 1):
        response = llm.chat(messages=messages, tools=tools_schema)

        if "text" in response and "tool_call" not in response:
            log.append({"step": step, "action": "final_answer",
                         "text": response["text"]})
            return {"answer": response["text"], "log": log,
                    "n_calls": llm.calls, "steps_used": step}

        call = response.get("tool_call")
        if not call:
            return {"answer": "(no answer)", "log": log, "n_calls": llm.calls,
                    "steps_used": step}

        # Execute the tool
        fn = tool_registry.get(call["name"])
        try:
            result = fn(**call["arguments"]) if fn else f"Unknown tool: {call['name']}"
        except Exception as e:
            result = f"Tool error: {type(e).__name__}: {e}"

        log.append({"step": step, "action": "tool_call",
                     "tool": call["name"], "arguments": call["arguments"],
                     "result": result})
        if verbose:
            print(f"  [step {step}] {call['name']}({call['arguments']}) → {str(result)[:120]}")

        # Append to the conversation so the LLM sees the result next turn
        messages.append({"role": "assistant", "content": "", "tool_call": call})
        messages.append({"role": "tool", "name": call["name"],
                          "content": json.dumps(result, default=str)})

    return {"answer": "(stopped: max_steps reached)", "log": log,
            "n_calls": llm.calls, "steps_used": max_steps}


# Try the agent on three different questions
for q in [
    "How many tickets in total?",
    "What is the mean satisfaction for Phone?",
    "Give me a channel summary for Chat.",
]:
    print(f"\n❓ {q}")
    out = run_agent(q, TOOLS, TOOL_FNS)
    print(f"💡 {out['answer']}")


> 💡 **`max_steps` is the most important parameter on an agent.** Without it, a model that keeps requesting tools (or a tool that produces results the model can't interpret) will loop forever. Budget defensively.

---

### ✋ Quick exercise (~2 min) — Run the agent loop

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Use `run_agent` to ask for the **mean satisfaction for Email**, then print the final answer and how many loop steps it took (`steps_used`). Reuse the `TOOLS` and `TOOL_FNS` you built earlier.

In [ ]:
# ✍️ Your turn 👇
# Call run_agent("...", TOOLS, TOOL_FNS) to ask about Email satisfaction,
# then print the answer and out["steps_used"].
# out = run_agent(...)

<details>
<summary>✅ <b>Solution</b></summary>

```python
out = run_agent("What is the mean satisfaction for Email?", TOOLS, TOOL_FNS)
print(out["answer"])
print("steps used:", out["steps_used"])
```

`run_agent` drives the call → execute → return loop and hands back a dict. `steps_used` is 2 here — one step to call the tool, one to turn the result into the final answer — well under the `max_steps` budget.
</details>

## 6. Inspecting the trace — debugging an agent

Production agents are debugged by reading their trace: the ordered list of *what the model decided*, *which tool ran*, and *what came back*. Our function returns the trace as `out["log"]`.

In [ ]:
out = run_agent("Tell me the summary for Email", TOOLS, TOOL_FNS, verbose=False)
print("Trace:")
for entry in out["log"]:
    print(f"  {entry}")
print(f"\nTotal LLM calls: {out['n_calls']}    Steps used: {out['steps_used']}")


## 7. A multi-step agent — calculator chain

What if a question needs *two* tool calls? Let's give the agent a request that combines numerical lookup with arithmetic.

(The MockLLM here is simple, so this is a demonstration. With a real LLM you'd see it reason and chain naturally.)

In [ ]:
# A tool that returns a number we can then feed into the calculator
out = run_agent("Compute 250 * 4", TOOLS, TOOL_FNS)
print(f"\nFinal: {out['answer']}")
print(f"Tool history: {[e.get('tool') for e in out['log'] if e['action']=='tool_call']}")


### The trace as a picture

Reading a trace as text works for three steps; production traces have dozens. Every serious agent platform (LangSmith, Langfuse, the OpenAI traces view) therefore draws the trace as a **timeline**: one box per step, tool executions on one lane, model decisions on another. Ours takes ~15 lines of matplotlib — and the same function works on *any* `run_agent` output:

In [ ]:
import matplotlib.pyplot as plt

def plot_trace(out, title="Agent trace"):
    log = out["log"]
    xs  = range(len(log))
    ys  = [0 if e["action"] == "tool_call" else 1 for e in log]

    fig, ax = plt.subplots(figsize=(2.2 * len(log) + 2, 2.8))
    ax.plot(xs, ys, color="lightgray", lw=2, zorder=1)        # the path through the loop
    for x, y, e in zip(xs, ys, log):
        is_tool = e["action"] == "tool_call"
        ax.scatter(x, y, s=700, marker="s", zorder=3,
                   color="#4c72b0" if is_tool else "#55a868")
        top    = (f"{e['tool']}({', '.join(str(v) for v in e['arguments'].values())})"
                  if is_tool else "final answer")
        bottom = str(e["result"] if is_tool else e["text"])[:30]
        ax.annotate(top,    (x, y), xytext=(0,  22), textcoords="offset points",
                    ha="center", fontsize=9, fontweight="bold")
        ax.annotate(bottom, (x, y), xytext=(0, -30), textcoords="offset points",
                    ha="center", fontsize=8, color="dimgray")
    ax.set(title=title, xticks=list(xs),
           xticklabels=[f"step {e['step']}" for e in log],
           yticks=[0, 1], yticklabels=["tool call", "LLM"],
           ylim=(-0.9, 1.9), xlim=(-0.5, len(log) - 0.5))
    plt.show()

plot_trace(out, title="Trace — 'Compute 250 * 4'")

---

### ✋ Quick exercise (~2 min) — Read the trace

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

From a finished run's `out["log"]`, build a list of `(tool_name, result)` tuples for each tool the agent actually called. Only `tool_call` entries name a tool — skip the final-answer entry.

```python
out = run_agent("Give me a channel summary for Chat.", TOOLS, TOOL_FNS, verbose=False)
```

In [ ]:
# ✍️ Your turn 👇
out = run_agent("Give me a channel summary for Chat.", TOOLS, TOOL_FNS, verbose=False)
# build: calls = list of (tool_name, result) for each tool_call entry in out["log"]


<details>
<summary>✅ <b>Solution</b></summary>

```python
out = run_agent("Give me a channel summary for Chat.", TOOLS, TOOL_FNS, verbose=False)
calls = [(e["tool"], e["result"]) for e in out["log"] if e["action"] == "tool_call"]
print(calls)
```

Every entry in `out["log"]` tags itself with an `action`. Filtering for `"tool_call"` and reading the `tool` and `result` fields reconstructs exactly what the *hands* did — the heart of debugging any agent.
</details>

## 8. When NOT to use an agent

Agents are powerful but expensive (multiple LLM calls per question, slow, harder to test). Don't reach for one when:

| Situation | Better choice |
|---|---|
| A single function would answer the user | Just expose the function, no LLM. |
| The user's question can be answered with one LLM call | Use a normal prompt. |
| The result must be exactly reproducible across runs | Code, not an LLM. |
| Latency must be under ~500 ms | Single call, not an agent loop. |

> 🎯 **A useful test.** If you can write the "if-this-then-that" logic in ~30 lines of Python, *write the 30 lines*. The LLM agent is for cases where the *decision tree* is too big or too fuzzy to enumerate.

## 9. Going live — the real-provider sketch

A real OpenAI / Anthropic tool-calling loop is almost identical to ours. Reference code:

```python
from openai import OpenAI
client = OpenAI()

def real_run(question, tools, tool_registry, max_steps=5):
    messages = [{"role": "system",  "content": "You are a data assistant."},
                {"role": "user",    "content": question}]
    for _ in range(max_steps):
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=[{"type": "function", "function": t} for t in tools],
        )
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return msg.content                              # final answer
        for tc in msg.tool_calls:
            fn = tool_registry[tc.function.name]
            args = json.loads(tc.function.arguments)
            result = fn(**args)
            messages.append({"role": "assistant", "tool_calls": [tc.model_dump()]})
            messages.append({"role": "tool", "tool_call_id": tc.id,
                              "content": json.dumps(result, default=str)})
    return "(max steps reached)"
```

Notice the shape: **same loop, same registry pattern, same safety budget.** Only the inside of the `client.chat.completions.create(...)` call differs. The investment you made in the mock pays off — your application code stays unchanged.

## 🧪 Practice exercises

### Exercise 1 — ⭐ Add a tool

Add a tool `cheapest_channel()` that returns the channel with the lowest `mean cost_per_ticket`. Register it in `TOOL_FNS`, add its schema to `TOOLS`, and update the MockLLM's keyword rule so that questions like *"which channel is cheapest?"* route to it.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def cheapest_channel() -> dict:
    g = support_ops.groupby("channel")["cost_per_ticket"].mean()
    name = g.idxmin()
    return {"channel": name, "mean_cost": round(float(g.min()), 2)}

TOOL_FNS["cheapest_channel"] = cheapest_channel
TOOLS.append({
    "name": "cheapest_channel",
    "description": "Return the channel with the lowest mean cost per ticket.",
    "parameters": {"type": "object", "properties": {}, "required": []},
})

# The offline MockLLM routes by keyword, so it won't auto-discover this new tool.
# With a REAL provider you do nothing extra — the model reads the tool's
# "description" and decides to call it. To see it work offline, call it directly:
print(cheapest_channel())
```

In the MockLLM's `_should_call`, add `"cheapest_channel": ["cheap", "lowest cost", "cost"]`. With a real LLM the rule disappears — the model reads the new tool's `description` and routes on its own.
</details>

### Exercise 2 — ⭐⭐ Pretty-print the trace

Write `print_trace(out)` that displays the agent log as a numbered list, with one line per step showing either the tool call or the final answer.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def print_trace(out):
    for entry in out["log"]:
        n = entry["step"]
        if entry["action"] == "tool_call":
            print(f"  {n}. CALL {entry['tool']}({entry['arguments']})")
            print(f"     → {str(entry['result'])[:100]}")
        else:
            print(f"  {n}. ANSWER: {entry['text']}")

out = run_agent("Mean satisfaction for Chat", TOOLS, TOOL_FNS, verbose=False)
print_trace(out)
```

A clean trace printer is the single best agent-debugging tool. Production teams build dashboards out of these traces.
</details>

### Exercise 3 — ⭐⭐ Add an out-of-tools guard

Modify `run_agent` so that if the model requests a tool that **isn't registered**, the function returns a clear error message instead of crashing. (Hint: the function already handles unknown tools partially — make sure it both reports cleanly *and* stops the loop.)

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def run_agent_safe(user_question, tools_schema, tool_registry, max_steps=5):
    messages = [{"role": "system",
                 "content": "You are a data assistant."},
                {"role": "user", "content": user_question}]
    log = []
    for step in range(1, max_steps + 1):
        resp = llm.chat(messages=messages, tools=tools_schema)
        if "text" in resp and "tool_call" not in resp:
            log.append({"step": step, "action": "final_answer", "text": resp["text"]})
            return {"answer": resp["text"], "log": log}

        call = resp.get("tool_call")
        if not call:
            return {"answer": "(no answer)", "log": log}
        fn = tool_registry.get(call["name"])
        if fn is None:
            log.append({"step": step, "action": "tool_missing",
                         "tool": call["name"]})
            return {"answer": f"Error: model asked for unknown tool {call['name']!r}.",
                    "log": log}
        result = fn(**call["arguments"])
        log.append({"step": step, "action": "tool_call",
                     "tool": call["name"], "arguments": call["arguments"],
                     "result": result})
        messages.append({"role": "assistant", "content": "", "tool_call": call})
        messages.append({"role": "tool", "name": call["name"],
                          "content": json.dumps(result, default=str)})
    return {"answer": "(stopped)", "log": log}
```

**Why this matters.** Production LLMs occasionally hallucinate tool names. Your code should fail clearly when that happens — *stopping the loop with a clear error* is much better than *silently returning garbage*.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

The function below tries to handle multiple tool calls per response — but it has a skip-every-other bug: half the calls are never executed. Find it.

```python
def buggy_handle(calls, registry):
    results = []
    i = 0
    while i < len(calls):
        result = registry[calls[i]["name"]](**calls[i]["arguments"])
        results.append(result)
        i = i + 2     # ← bug!
    return results
```

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
# Your fixed version  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def handle(calls, registry):
    return [registry[c["name"]](**c["arguments"]) for c in calls]
```

The original was incrementing `i` by 2 each iteration, skipping every other call. The fix is `i += 1` (or, more Pythonically, just use a `for` loop / comprehension). This is the classic manual-index loop bug from NB 2 — Python rewards `for` loops over manual indices.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise C — ⭐⭐⭐ Two-tool agent: calculator + lookup

Extend the agent loop with **two** tools:

- `add(a, b)`
- `lookup_price(product)` — returns a dict like `{"price": 99.0}` for known products.

Given the user query *'How much do an apple and a banana cost together?'* the agent should call `lookup_price("apple")`, then `lookup_price("banana")`, then `add(...)` with the two prices, and finally print the total.

For this offline exercise you can call the tools by hand — just demonstrate the routing pattern.

In [ ]:
# Your code here  👇
PRICES = {"apple": 1.0, "banana": 0.5, "cherry": 3.0}

def add(a, b):              return a + b
def lookup_price(product):  return {"price": PRICES[product]}

TOOL_MAP = {"add": add, "lookup_price": lookup_price}

# Mock a 3-step agent trace as a list of (tool_name, args) tuples.
# Then execute them in order and print the result.
# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
PRICES = {"apple": 1.0, "banana": 0.5, "cherry": 3.0}

def add(a, b):              return a + b
def lookup_price(product):  return {"price": PRICES[product]}

TOOL_MAP = {"add": add, "lookup_price": lookup_price}

trace = [
    ("lookup_price", {"product": "apple"}),
    ("lookup_price", {"product": "banana"}),
    ("add",          "__use_previous_two_prices__"),
]

memory = []
for name, args in trace:
    if name == "add":
        a, b = memory[-2]["price"], memory[-1]["price"]
        result = TOOL_MAP[name](a, b)
    else:
        result = TOOL_MAP[name](**args)
    memory.append(result)
    print(f"  → {name}({args})  ⇒  {result}")

print(f"\nTotal: ${memory[-1]:.2f}")
```

**Reasoning.** The agent loop's hardest design question isn't *which tools to expose* — it's **how the agent passes results between steps**. Three options you'll see in real frameworks. (1) **A scratchpad / memory list** (what we did) — simple, transparent, and the way ReAct-style agents work. (2) **Named variables** — the model says 'store this as `apple_price`', then references the name later (LangChain agents work this way). (3) **Implicit composition** via the LLM rewriting earlier results into the next prompt. Most production agents use a combination. The mock trace here makes the dataflow explicit — that's worth doing before you rely on an LLM to do it right.
</details>

### Stretch exercise D — ⭐⭐⭐ Add a 'refuse' tool

Production agents need a way to **decline** when the user asks for something they shouldn't do. Add a third tool `refuse(reason)` that the agent uses when a query is out of scope (e.g. medical advice from a customer-support bot).

Demonstrate the agent routing the query *'should I take ibuprofen for my headache?'* to `refuse` rather than to the LLM.

In [ ]:
# Your code here  👇
def lookup_price(product): return {"price": 1.0}
def add(a, b):             return a + b
def refuse(reason):        return {"refused": True, "reason": reason}

TOOL_MAP = {"lookup_price": lookup_price, "add": add, "refuse": refuse}

OUT_OF_SCOPE = {"medical", "legal", "financial advice", "diagnose"}

def route(query):
    ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def lookup_price(product): return {"price": 1.0}
def add(a, b):             return a + b
def refuse(reason):        return {"refused": True, "reason": reason}

TOOL_MAP = {"lookup_price": lookup_price, "add": add, "refuse": refuse}

OUT_OF_SCOPE = {"ibuprofen", "medical", "diagnose", "prescription", "legal", "lawyer"}

def route(query):
    q = query.lower()
    if any(w in q for w in OUT_OF_SCOPE):
        return TOOL_MAP["refuse"](
            "This bot only handles product/billing questions — please ask a qualified professional."
        )
    if "price" in q or "cost" in q:
        return TOOL_MAP["lookup_price"](product="apple")
    return TOOL_MAP["refuse"]("I'm not sure how to handle that.")

print(route("should I take ibuprofen for my headache?"))
print(route("what's the price of apples?"))
```

**Reasoning.** Refusal is a first-class tool, not an exception. Three reasons. (1) **Auditability** — every refusal is logged with a structured reason, which is what your compliance team is going to ask for. (2) **Consistency** — the agent always returns the same shape (`{refused: bool, ...}`), so downstream code doesn't branch on string content. (3) **Testability** — you can unit-test the routing rules without ever calling the LLM. In a real system you'd also have the LLM itself produce a `refuse(reason)` call when *it* doesn't want to comply, so the routing logic and the model's safety judgement converge on the same structured output.
</details>

## 🧠 Key takeaways

We set out to build a support-ops analyst that answers questions on its own — and we did, in about a hundred lines. The whole thing was just the **planner and the hands** passing notes:

1. **Tool calling = the LLM picks; your code runs.** The planner asks for `mean_satisfaction(channel='Chat')`; the hands actually run it. The LLM never executes anything itself.
2. Tools are described with a **JSON schema** — name, description, argument types. That's the menu you hand the planner.
3. The agent loop is **5 lines**: call LLM → got tool_call? → run tool → append result → loop.
4. **Always set `max_steps`.** An agent without a budget is a kernel waiting to hang.
5. **Log every call.** The trace is your only debugging tool when an agent misbehaves — read it and ask "was that the planner or the hands?"
6. **When unsure, code it.** Agents shine when the decision tree is too big to write by hand.
7. The shape of every tool-calling API (OpenAI / Anthropic / etc.) is identical — your code is portable.
8. A single `run_pandas_query` or `run_sql` tool turns an LLM into a real **data assistant** — exactly the one we built.

> 🧭 **The mental model, one more time.** An agent is a planner that *asks* and hands that *act*, looping until the planner is satisfied. Bigger agents add more tools and more steps, but it's always those two characters passing notes — and `max_steps` is the leash that keeps the conversation from running forever.

## ✅ Self-assessment

- [ ] Write a JSON-schema tool definition that a model could call
- [ ] Run the call → execute → return loop manually
- [ ] Build a `run_agent` function with `max_steps` and a tool registry
- [ ] Read an agent trace and identify which tool ran with which arguments
- [ ] Handle the case where the model requests an unknown tool
- [ ] Explain when an agent is worth it versus a single prompt

## 🚀 Next step

Continue with **Notebook 13 (fast track) — From Notebook to Project**, where you turn throwaway notebook code (like this agent) into an installable, tested Python package — the last step before shipping.